# Convert COMSOL sensitivity files to XLSX files with horizontal voltages
## For Res2DInv DAT file generation.

## Define utility functions


In [ ]:
from pathlib import Path

"""Reshape a sensitivity-analysis CSV into the horizontal layout used by
average_trace_all_first_600um_ML_decoded.xlsx: four blank columns
followed by V_0..V_n voltage values in a single row.
"""

def csv_to_horizontal_xlsx(csv_path: str) -> None:
    # convert to a path object
    csv_path = Path(csv_path)

    # open file and read data
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        reader.fieldnames = [n.strip() for n in reader.fieldnames]
        rows = list(reader)

    # define columns map
    columns = {"plus": "plus_500_Ohmcm", "minus": "minus_500_Ohmcm", "V_set": "base"}

    vset = [float(row["V_set"]) for row in rows]

    # save each column to a new file copying the values horizontally
    for field, suffix in columns.items():
        if field == "plus":
            values = [v + float(row[field]) for v, row in zip(vset, rows)]
        elif field == "minus":
            values = [v - float(row[field]) for v, row in zip(vset, rows)]
        else:
            values = [float(row[field]) for row in rows]

        header = ["blank_4", "blank_3", "blank_2", "blank_1"] + [f"V_{i}" for i in range(len(values))]

        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "Sheet1"
        ws.append(header)
        ws.append([None, None, None, None, *values])

        out_path = csv_path.with_name(f"{csv_path.stem}_{suffix}.xlsx")
        wb.save(out_path)


"""Reshape a per-row sensitivity-analysis CSV into the horizontal layout used by
average_trace_all_first_600um_ML_decoded.xlsx: four blank columns
followed by the first n_voltages V_0..V_n values in a single row, one output
file per CSV row (named from that row's `file_name` column).
"""

def comsol_csv_to_horizontal_xlsx(csv_path: str, n_voltages: int = 18) -> None:
    # convert to a path object
    csv_path = Path(csv_path)

    # open file and read data
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        reader.fieldnames = [n.strip() for n in reader.fieldnames]
        rows = list(reader)

    # identify voltage columns (V_0, V_1, ...), sorted by their numeric index
    voltage_fields = sorted(
        (name for name in reader.fieldnames if re.fullmatch(r"V_\d+", name)),
        key=lambda name: int(name.split("_")[1]),
    )[:n_voltages]

    # each row names its own output file; write its first n_voltages values horizontally
    for row in rows:
        file_name = row["file_name"].strip()
        values = [float(row[field]) for field in voltage_fields]

        header = ["blank_4", "blank_3", "blank_2", "blank_1"] + [f"V_{i}" for i in range(len(values))]

        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "Sheet1"
        ws.append(header)
        ws.append([None, None, None, None, *values])

        out_path = csv_path.with_name(f"{csv_path.stem}_{file_name}.xlsx")
        wb.save(out_path)

## Run conversion from processed column files


In [ ]:
root_folder = (r".../original_orientation")

list_of_csvs = [
    "sensitivity_analysis_comsol_GCL.csv",
    "sensitivity_analysis_comsol_IPL.csv",
	"sensitivity_analysis_comsol_Nu.csv",
	"sensitivity_analysis_comsol_PR.csv",
]

for csv_path in list_of_csvs:

    # concat root folder with file name
    file_path = Path(root_folder) / csv_path

	# run conversion
    csv_to_horizontal_xlsx(file_path)

## Run conversion from raw COMSOL CSV files


In [ ]:
root_folder = (r".../original_orientation")

list_of_csvs = [
    "sensitivity_analysis_comsol_250_Ohmcm.csv"
]

for csv_path in list_of_csvs:

    # concat root folder with file name
    file_path = Path(root_folder) / csv_path

	# run conversion
    comsol_csv_to_horizontal_xlsx(file_path)

# Convert COMSOL CSV files to XLSX files
## For Res2DInv DAT file generation.

## Define utility functions

In [ ]:
import re
import csv
import openpyxl
from pathlib import Path

"""Reshape a per-row sensitivity-analysis CSV into the horizontal layout used by
average_trace_all_first_600um_ML_decoded.xlsx: four blank columns
followed by the first n_voltages V_0..V_n values in a single row, one output
file per CSV row (named from that row's leading resistivity columns).
"""

def comsol_csv_with_multiple_rows_by_resistivity_to_horizontal_xlsx(csv_path: str, output_folder: str = None,
																	n_voltages: int = 18, output_prefix: str = "") -> None:

    # convert to a path object
    output_folder = Path(output_folder)
    csv_path = Path(csv_path)

    # create output folder if it doesn't exist and different from the input folder; otherwise use the input folder
    if output_folder is not None:
        output_folder.mkdir(parents=True, exist_ok=True)
    else:
	    output_folder = csv_path.parent

    # open file and read data
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        reader.fieldnames = [n.strip() for n in reader.fieldnames]
        rows = list(reader)

    # identify voltage columns (V_0, V_1, ...), sorted by their numeric index
    voltage_fields = sorted(
        (name for name in reader.fieldnames if re.fullmatch(r"V_\d+", name)),
        key=lambda name: int(name.split("_")[1]),
    )[:n_voltages]

    # everything else is a resistivity column, in the order it appears in the header
    resistivity_fields = [name for name in reader.fieldnames if not re.fullmatch(r"V_\d+", name)]

    # each row names its own output file from its resistivity values; write its
    # first n_voltages values horizontally
    for row in rows:
        file_name = output_prefix + "_".join(
            f"{field.removeprefix('rho_')}_{row[field].strip()}" for field in resistivity_fields
        )
        values = [float(row[field]) for field in voltage_fields]

        header = ["blank_4", "blank_3", "blank_2", "blank_1"] + [f"V_{i}" for i in range(len(values))]

        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "Sheet1"
        ws.append(header)
        ws.append([None, None, None, None, *values])

        out_path = output_folder / f"{file_name}.xlsx"
        wb.save(out_path)

## Run conversion (this is a scratch pad; do not rely on values)

In [ ]:
root_folder = (r".../comsol_output_forward_traces")

list_of_csvs = [
    "base_model_with_more_conductive_pr_and_gcl_33.csv"
]

for csv_path in list_of_csvs:

    # concat root folder with file name
    file_path = Path(root_folder) / csv_path

	# run conversion
    comsol_csv_with_multiple_rows_by_resistivity_to_horizontal_xlsx(file_path)

## Convert LE test dataset CSV file

In [ ]:
root_folder = (r".../test_data_le")
list_of_csvs = ["MC_data_out_33_N=8000.csv"]
output_prefix = "comsol_testset_8000_"
output_folder = Path(r".../res2dinv_format")

for csv_path in list_of_csvs:

    # concat root folder with file name
    file_path = Path(root_folder) / csv_path

	# run conversion
    comsol_csv_with_multiple_rows_by_resistivity_to_horizontal_xlsx(file_path, output_prefix=output_prefix, output_folder=output_folder)

## Convert RCS test dataset CSV file

In [ ]:
root_folder = (r".../test_data_rcs")
list_of_csvs = ["MC_data_out_33_N=8000_RCS.csv"]
output_prefix = "comsol_testset_8000_"
output_folder = Path(r".../res2dinv_format")

for csv_path in list_of_csvs:

    # concat root folder with file name
    file_path = Path(root_folder) / csv_path

	# run conversion
    comsol_csv_with_multiple_rows_by_resistivity_to_horizontal_xlsx(file_path, output_prefix=output_prefix, output_folder=output_folder)